# 04 - Out-of-time check

This is a new group of loans: 2006 Q1 originations, eligible at the December 2007 snapshot, with outcomes in 2008. I use their origination fields and apply the scorecard fitted on the 2005 Q1 development loans. Nothing is refitted here.

Some loans have no reliable 2008 outcome. They still receive a PD, but stay out of the performance numbers.


In [1]:
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from cohort import build_cohort, NUMERIC

LOCAL_DATA = Path(r"E:\CRM Projects\Data")
if "POCKET_CRM_DATA" not in os.environ and LOCAL_DATA.is_dir():
    os.environ["POCKET_CRM_DATA"] = str(LOCAL_DATA)
DATA = Path(os.environ.get("POCKET_CRM_DATA", "./data")).expanduser().resolve()
OUT = DATA / "pocket1_model"
champion = joblib.load(OUT / "woe_logistic_champion.joblib")
TARGET = "default_180_plus_2008"
features, selected = champion["features"], champion["selected"]


In [2]:
# The raw 2006 Q1 files may be inside an extracted subfolder.
source_dirs = [DATA] + [p.parent for p in DATA.rglob("perf_2006Q1.txt")]
source_dirs = list(dict.fromkeys(source_dirs))
source_dirs = [d for d in source_dirs if (d / "perf_2006Q1.txt").is_file()
               and (d / "orig_2006Q1.txt").is_file()]
if not source_dirs:
    raise FileNotFoundError(
        f"Need orig_2006Q1.txt and perf_2006Q1.txt under {DATA} for the 2008 test"
    )
if len(source_dirs) > 1:
    raise ValueError(f"More than one 2006 Q1 source folder: {source_dirs}")

data = build_cohort(source_dirs[0], "2006Q1", "200712", 2008)
X = data[features].copy()
X[NUMERIC] = X[NUMERIC].astype("float64")
cats = [f for f in features if f not in NUMERIC]
X[cats] = X[cats].astype(object).where(X[cats].notna(), np.nan)
print("Later-snapshot eligible loans:", len(data))


Later-snapshot eligible loans: 253343


In [3]:
# Old bins and coefficients, new loans.
bp, model = champion["binning_process"], champion["model"]
w = pd.DataFrame(np.asarray(bp.transform(X, **champion["woe_options"])),
                 index=X.index, columns=features)
for f, kind in champion["neutral_woe_bins"]:
    mask = (X[f].isna() if kind == "missing" else
        (X[f].eq(999).fillna(False) if f == "dti" else pd.Series(False, index=X.index)))
    w.loc[mask.to_numpy(dtype=bool), f] = 0.0

scored = data[["loan_id", TARGET]].copy()
scored["PD"] = model.predict_proba(w[selected])[:, 1]
scored["score"] = champion["offset"] - champion["factor"] * model.decision_function(w[selected])
print("Scored loans:", len(scored), "| verified answers:", scored[TARGET].notna().sum())


Scored loans: 253343 | verified answers: 253094


In [4]:
# Assess only loans with a verified 2008 answer.
eval_ = scored.loc[scored[TARGET].notna()].copy()
actual = eval_[TARGET].astype(int)

if actual.nunique() != 2:
    raise ValueError("Need both verified defaults and nondefaults in 2008")
fpr, tpr, _ = roc_curve(actual, eval_["PD"])
auc = roc_auc_score(actual, eval_["PD"])
metrics = pd.DataFrame([dict(loans=len(eval_), defaults=actual.sum(),
    excluded=len(scored)-len(eval_), AUC=auc, KS=(tpr-fpr).max(), Gini=2*auc-1,
    Brier=brier_score_loss(actual, eval_["PD"]), observed=actual.mean(),
    predicted=eval_["PD"].mean())])
eval_["decile"] = pd.qcut(-eval_["PD"], 10, labels=False, duplicates="drop") + 1
calibration = eval_.groupby("decile").agg(loans=(TARGET, "size"),
    defaults=(TARGET, "sum"), observed_PD=(TARGET, "mean"), predicted_PD=("PD", "mean"))
print(metrics.round(4).to_string(index=False))
print("Decile 1 = highest risk:\n", calibration.round(4))


 loans  defaults  excluded    AUC     KS   Gini  Brier  observed  predicted
253094      1975       249 0.7623 0.3917 0.5247 0.0077    0.0078     0.0033
Decile 1 = highest risk:
         loans  defaults  observed_PD  predicted_PD
decile                                            
1       25332       611       0.0241        0.0124
2       25424       432        0.017        0.0064
3       25178       301        0.012        0.0044
4       25353       204        0.008        0.0031
5       25403       175       0.0069        0.0023
6       25169       116       0.0046        0.0016
7       25310        90       0.0036        0.0011
8       25543        29       0.0011        0.0008
9       25161        10       0.0004        0.0005
10      25221         7       0.0003        0.0002


In [5]:
# Compare train and 2008 bin shares with PSI.
train_bins = pd.read_parquet(OUT / "bins_train.parquet")
oot_bins = pd.DataFrame(np.asarray(bp.transform(X, metric="bins")),
                        columns=features).astype("string")
psi_rows = []
for f in selected:
    p = train_bins[f].value_counts(normalize=True, dropna=False)
    q = oot_bins[f].value_counts(normalize=True, dropna=False)
    labels = p.index.union(q.index)

    p = p.reindex(labels, fill_value=0).clip(lower=1e-6)
    q = q.reindex(labels, fill_value=0).clip(lower=1e-6)
    psi_rows.append({"feature":f, "PSI":((q-p)*np.log(q/p)).sum()})
psi = pd.DataFrame(psi_rows).sort_values("PSI", ascending=False)
print("Bin drift:\n", psi.round(4).to_string(index=False))
data.to_parquet(OUT / "oot_2006q1_dec2007_2008_raw.parquet", index=False)
scored.to_parquet(OUT / "oot_2006q1_dec2007_2008_scored.parquet", index=False)
metrics.to_csv(OUT / "oot_2008_metrics.csv", index=False)
calibration.to_csv(OUT / "oot_2008_calibration.csv")
psi.to_csv(OUT / "oot_2008_psi.csv", index=False)


Bin drift:
                feature    PSI
original_interest_rate 2.5877
           seller_name 2.1735
    original_loan_term 0.0902
          original_upb 0.0115
                   dti 0.0103
                   ltv 0.0093
                  fico 0.0015
